# 03: Linear 層を SVD で 2 層に置き換える

## やること
1. 元の `Linear(784 → 512)` の重み `W` を SVD する
2. rank `r=64` で打ち切る
3. 1 層を **2 層** に置き換える
   - `Linear(784 → r, bias=False)` ← 重みに `Vh_r`
   - `Linear(r → 512, bias=True)` ← 重みに `U_r @ diag(S_r)`、bias は元の `b`
4. 出力誤差とパラメータ数（圧縮率）を確認する

## 01 / 02 との違い
| | 01 | 02 | **03（このノート）** |
|---|---|---|---|
| 近似のやり方 | `W_r` を行列として再構成 | 同じ（rank を複数比較） | **実際に 2 つの Linear 層にする** |
| 目的 | 誤差を体感 | トレードオフ表 | NN として置き換え可能か確認 |

数式の対応:
`W ≈ U_r @ diag(S_r) @ Vh_r`
→ `y ≈ x @ Vh_r.T @ (U_r @ diag(S_r)).T + b`
→ `y ≈ second(first(x))`

In [1]:
import torch
import torch.nn as nn

# ============================================================
# ステップ1: 元の Linear 層を用意し、基準となる出力を取る
# ============================================================

torch.manual_seed(42)  # 01 / 02 と同じ seed → 同じ結果と比較できる

# MNIST 想定: 784次元 → 512次元の全結合層（これが「圧縮前」）
original_layer = nn.Linear(784, 512)

# ダミー入力: 32枚 × 784次元
x = torch.randn(32, 784)

# 元の出力（あとで 2 層版の出力と比較する基準）
y_original = original_layer(x)  # (32, 512)

# PyTorch の weight は (out, in) = (512, 784)
W = original_layer.weight.data  # (512, 784)
b = original_layer.bias.data    # (512,)

print("W shape:", W.shape)
print("b shape:", b.shape)
print("y_original shape:", y_original.shape)

W shape: torch.Size([512, 784])
b shape: torch.Size([512])
y_original shape: torch.Size([32, 512])


In [2]:
# ============================================================
# ステップ2: SVD → rank r で打ち切り
# ============================================================
# W ≈ U @ diag(S) @ Vh
# 上位 r 個だけ残す → W_r ≈ U_r @ diag(S_r) @ Vh_r

U, S, Vh = torch.linalg.svd(W, full_matrices=False)

r = 64  # 使う特異値の個数（小さいほど圧縮↑・誤差↑）

U_r = U[:, :r]    # (512, 64)  出力側の方向
S_r = S[:r]       # (64,)      特異値（重要度）
Vh_r = Vh[:r, :]  # (64, 784)  入力側の方向

print("U_r shape:", U_r.shape)
print("S_r shape:", S_r.shape)
print("Vh_r shape:", Vh_r.shape)
# 次のセルで、これらを「2 つの Linear の重み」に埋め込む

U_r shape: torch.Size([512, 64])
S_r shape: torch.Size([64])
Vh_r shape: torch.Size([64, 784])


In [3]:
# ============================================================
# ステップ3: 1 層 → 2 層に置き換える（ここが 03 の本題）
# ============================================================
# 元:   Linear(784 → 512)
# 後:   Linear(784 → r, bias=False)  +  Linear(r → 512, bias=True)
#
# なぜこう分解できるか:
#   y = x @ W.T + b
#     ≈ x @ (U_r @ diag(S_r) @ Vh_r).T + b
#     = x @ Vh_r.T @ (U_r @ diag(S_r)).T + b
#
# PyTorch の Linear は内部で y = x @ weight.T + bias なので
#   first.weight  = Vh_r              → (r, 784)
#   second.weight = U_r @ diag(S_r)   → (512, r)

first_layer = nn.Linear(784, r, bias=False)  # 中間次元 r、bias は不要
second_layer = nn.Linear(r, 512, bias=True)  # 最終出力 512、bias はここに置く

# 学習ではなく「SVD の結果を手で書き込む」
first_layer.weight.data = Vh_r                    # (64, 784)
second_layer.weight.data = U_r @ torch.diag(S_r)  # (512, 64)
second_layer.bias.data = b                        # 元の bias をそのまま

# 2 層を順番に通す = 近似した 1 層を通すのと同じ計算
y_two_layer = second_layer(first_layer(x))  # (32, 512)

print("first.weight:", first_layer.weight.shape)
print("second.weight:", second_layer.weight.shape)
print("y_two_layer shape:", y_two_layer.shape)

first.weight: torch.Size([64, 784])
second.weight: torch.Size([512, 64])
y_two_layer shape: torch.Size([32, 512])


In [4]:
# ============================================================
# ステップ4: 元の 1 層出力と、2 層版の出力を比較
# ============================================================
# 01 で W_r 行列を使ったときの MSE と、ほぼ同じ値になるはず
# （やり方が違うだけで、数学的には同じ近似）

diff = y_original - y_two_layer

mae = torch.mean(torch.abs(diff))           # 平均絶対誤差
mse = torch.mean(diff ** 2)                 # 二乗平均誤差
max_abs_error = torch.max(torch.abs(diff))  # 最大ズレ

print("MAE:", mae.item())
print("MSE:", mse.item())
print("Max abs error:", max_abs_error.item())
# rank=64 なので 0 にはならない（情報を捨てているため）

MAE: 0.38189417123794556
MSE: 0.22959685325622559
Max abs error: 1.961763620376587


In [5]:
# ============================================================
# ステップ5: パラメータ数と圧縮率を確認
# ============================================================
# 元:   W(512×784) + b(512) = 401,920
# 圧縮: Vh_r(64×784) + (U_r@diag(S_r))(512×64) + b(512)
#     = 50,176 + 32,768 + 512 = 83,456
#
# compression_ratio = 元 / 圧縮後
#   > 1 → 圧縮できている（rank=64 なら約 4.8 倍）

original_params = W.numel() + b.numel()

compressed_params = (
    sum(p.numel() for p in first_layer.parameters())
    + sum(p.numel() for p in second_layer.parameters())
)

compression_ratio = original_params / compressed_params

print("Original params:", original_params)
print("Compressed params:", compressed_params)
print("Compression ratio:", compression_ratio)
# → 約 4.8 倍に圧縮できている、という意味

Original params: 401920
Compressed params: 83456
Compression ratio: 4.815950920245399
